### TUGAS DATA MINING SEQUENTIAL PATTERN MINING
Jonathan Joyo Wibowo

5054251026

Rekayasa Kecerdasan Artifisial

### SETUP

In [1]:
!pip install prefixspan -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import warnings
from google.colab import drive

from prefixspan import PrefixSpan
SEED = 67

In [3]:
warnings.filterwarnings('ignore', category=DeprecationWarning)

In [4]:
uploaded = drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
pd.set_option('display.width', 1000)
pd.set_option('display.max_columns', None)

### LOAD DATA

In [6]:
df_data = pd.read_excel("/content/drive/MyDrive/dataset/online_retail/Online_Retail.xlsx")

In [7]:
df_data_raw = df_data.copy()

### EDA

In [8]:
df_data.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [9]:
df_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [10]:
df_data.nunique()

,0
InvoiceNo,25900
StockCode,4070
Description,4223
Quantity,722
InvoiceDate,23260
UnitPrice,1630
CustomerID,4372
Country,38


In [11]:
df_data.shape

(541909, 8)

In [12]:
df_data["Quantity"].describe()

,Quantity
count,541909.000000
mean,9.552250
std,218.081158
min,-80995.000000
25%,1.000000
50%,3.000000
75%,10.000000
max,80995.000000


In [13]:
df_data[df_data["Quantity"] < 1].count()

,0
InvoiceNo,10624
StockCode,10624
Description,9762
Quantity,10624
InvoiceDate,10624
UnitPrice,10624
CustomerID,8905
Country,10624


### Preprocessing

In [14]:
df_data = df_data_raw.copy()

In [15]:
# Hapus duplikat
df_data = df_data.drop_duplicates()

# Hapus duplikat pada subset InvoiceNo, StockCode
# Karena kita tidak peduli seberapa banyak suatu barang ada pada suatu basket/transaksi
# Kita hanya peduli apakah suatu barang ADA atau TIDAK

df_data = df_data.drop_duplicates(subset=["InvoiceNo", "StockCode"])

In [16]:
# Hapus data yang quantity <= 0
df_data = df_data[df_data["Quantity"] > 0]
df_data = df_data.reset_index(drop=True)

# Hapus juga yang quantity >= 80000 (tidak normal)
df_data = df_data[df_data["Quantity"] < 80000]
df_data = df_data.reset_index(drop=True)

In [17]:
# Hapus baris dengan CustomerID kosong
df_data = df_data.dropna(subset=["CustomerID"])
df_data = df_data.reset_index(drop=True)

In [18]:
df_data.shape

(387874, 8)

In [19]:
# Hapus transaksi yang di-cancel
df_data["InvoiceNo"] = df_data["InvoiceNo"].astype(str)
cancelled_mask = df_data["InvoiceNo"].str.contains(r"^C", case=False, na=False)
df_data = df_data[~cancelled_mask]

In [20]:
df_data.shape

(387874, 8)

In [21]:
# drop kolom yang tidak perlu

df_data = df_data.drop(columns=["Description", "Quantity", "UnitPrice", "Country"], axis=1)

In [22]:
df_data.head()

,InvoiceNo,StockCode,InvoiceDate,CustomerID
0,536365,85123A,2010-12-01 08:26:00,17850.0
1,536365,71053,2010-12-01 08:26:00,17850.0
2,536365,84406B,2010-12-01 08:26:00,17850.0
3,536365,84029G,2010-12-01 08:26:00,17850.0
4,536365,84029E,2010-12-01 08:26:00,17850.0


In [23]:
# Urutkan data berdasarkan CustomerID dan InvoiceDate
df_data = df_data.sort_values(by=["CustomerID", "InvoiceDate"])

In [24]:
sequences_dict = dict()

for customer_id, group in df_data.groupby("CustomerID"):
    customer_sequences = list()

    for invoice_no, inv_group in group.groupby("InvoiceNo"):
        itemset = tuple(inv_group["StockCode"].unique())
        customer_sequences.append(itemset)

    sequences_dict[customer_id] = customer_sequences

dataset_sequences = list(sequences_dict.values())

In [43]:
# Opsional, buat dataframe yang isinya StockCode & Description
# nanti bisa dipake buat lookup kira-kira StockCode yang muncul
# di rule association itu produk apa

df_lookup = df_data_raw.copy()
df_lookup = df_lookup[["StockCode", "Description"]]

# drop duplikat
df_lookup = df_lookup.drop_duplicates()

# convert StockCode ke string
df_lookup["StockCode"] = df_lookup["StockCode"].astype(str)

# hapus yang duplikat
df_lookup = df_lookup.drop_duplicates(subset=["StockCode"])

### MODELING

In [52]:
# di sini saya pake 0.0025 soalnya waktu 0.01 itu cuma ada 2 pola yg memenuhi,
# sedangkan kalo 0.05 cuma ada 3 pola,
# kalo 0.0025 baru ada cukup banyak pola, yaitu 11 pola

total_customers = len(dataset_sequences)
min_sup_count = int(total_customers * 0.001)

print(f"Jumlah Customer: {total_customers}")
print(f"Minimum Support (1%): {min_sup_count} pelanggan")

Jumlah Customer: 4339
Minimum Support (1%): 4 pelanggan


In [54]:
ps = PrefixSpan(dataset_sequences)

freq_patterns = ps.frequent(min_sup_count)
freq_patterns.sort(key=lambda x: x[0], reverse=True)

print(f"Jumlah Sequential Patterns: {len(freq_patterns)}")
for sup, pattern in freq_patterns:
  print(f"Support: {sup} ({sup/total_customers*100:.2f}%) | Pattern: {pattern}")

Jumlah Sequential Patterns: 70
Support: 49 (1.13%) | Pattern: [('M',)]
Support: 45 (1.04%) | Pattern: [('POST',)]
Support: 22 (0.51%) | Pattern: [(23084,)]
Support: 13 (0.30%) | Pattern: [(22423,)]
Support: 12 (0.28%) | Pattern: [(21137,)]
Support: 12 (0.28%) | Pattern: [(23284,)]
Support: 10 (0.23%) | Pattern: [('M',), ('M',)]
Support: 10 (0.23%) | Pattern: [(21621,)]
Support: 10 (0.23%) | Pattern: [(47566,)]
Support: 10 (0.23%) | Pattern: [('BANK CHARGES',)]
Support: 10 (0.23%) | Pattern: [('85099B',)]
Support: 9 (0.21%) | Pattern: [(22720,)]
Support: 9 (0.21%) | Pattern: [(21181,)]
Support: 9 (0.21%) | Pattern: [(79321,)]
Support: 8 (0.18%) | Pattern: [(22961,)]
Support: 8 (0.18%) | Pattern: [('46000S',)]
Support: 8 (0.18%) | Pattern: [('85123A',)]
Support: 7 (0.16%) | Pattern: [('POST',), ('POST',)]
Support: 7 (0.16%) | Pattern: [(22722,)]
Support: 7 (0.16%) | Pattern: [(15036,)]
Support: 7 (0.16%) | Pattern: [(22178,)]
Support: 6 (0.14%) | Pattern: [(21034,)]
Support: 6 (0.14%) | 

In [64]:
seq_patterns = dict()

In [69]:
# kalo diliat dari cell sebelumnya, memang ada pattern yang supportnya jauh lebih tinggi
# tapi pattern-pattern tersebut panjangnya cuma 1, jadi ngga bisa dihitung confidencenya
# dan kalo panjangnya 1 itu bukan urutan
# jadi di sini saya paksa buat cari pattern yang panjangnya dua atau lebih

ps = PrefixSpan(dataset_sequences)

freq_patterns = ps.frequent(min_sup_count)
freq_patterns_filtered = [
    (sup, pattern) for sup, pattern in freq_patterns if len(pattern) >= 2
]
freq_patterns_filtered.sort(key=lambda x: x[0], reverse=True)

print(f"Jumlah Sequential Patterns: {len(freq_patterns_filtered)}")
for sup, pattern in freq_patterns_filtered:
  print(f"Support: {sup} ({sup/total_customers*100:.2f}%) | Pattern: {pattern}")


for i in range(3):
  sup, pattern = freq_patterns_filtered[i]
  seq_patterns[str(pattern[0][0])] = {"support": sup}

Jumlah Sequential Patterns: 7
Support: 10 (0.23%) | Pattern: [('M',), ('M',)]
Support: 7 (0.16%) | Pattern: [('POST',), ('POST',)]
Support: 6 (0.14%) | Pattern: [(79321,), (79321,)]
Support: 5 (0.12%) | Pattern: [(23084,), (23084,)]
Support: 4 (0.09%) | Pattern: [('M',), ('M',), ('M',)]
Support: 4 (0.09%) | Pattern: [(22423,), (22423,)]
Support: 4 (0.09%) | Pattern: [('85123A',), ('85123A',)]


In [79]:
supports = {"M": [49, 10],
            "POST": [45, 7],
            "79321": [9, 6]}

for pattern, support in supports.items():
  confidence = support[1] / support[0] * 100
  seq_patterns[pattern]["confidence"] = confidence
  seq_patterns[pattern]["lift"] = confidence / (support[0] / total_customers * 100)
  seq_patterns[pattern]["support_percentage"] = support[0] / total_customers * 100

In [80]:
pd.DataFrame(seq_patterns).T.sort_values(by="support", ascending=False)

,support,confidence,lift,support_percentage
M,10.0,20.408163,18.071637,1.129292
POST,7.0,15.555556,14.999012,1.037105
79321,6.0,66.666667,321.407407,0.207421


In [81]:
stock_codes = ["M", "POST", "79321"]

df_lookup[df_lookup["StockCode"].isin(stock_codes)]

,StockCode,Description
45,POST,POSTAGE
178,79321,CHILLI LIGHTS
2239,M,Manual


### INTERPRETASI

Ada 3 Aturan dengan skor support tertinggi, yaitu:
1. Manual (M) -> Manual (M)
2. Postage (POST) -> Postage (POST)
3. Chilli Lights (79321) -> Chilli Lights (79321)

Aturan pertama itu Confidence-nya 20.4%, berarti jika Manual (M) dibeli, maka kemungkinan Manual (M) dibeli lagi itu 20.4%.

Lift-nya 18.07, berarti jika Manual (M) dibeli, maka kemungkinan buat Manual (M) dibeli itu naik 18.07 kali lipat, yang awalnya 1.13% (Antecedent), naik jadi 20.4%.

Aturan kedua itu Confidence-nya 15.6%, berarti jika Postage (POST) dibeli, maka kemungkinan Postage (POST) dibeli lagi itu 15.6%.

Lift-nya 15.00, berarti jika Postage (POST) dibeli, maka kemungkinan buat Postage (POST) dibeli itu naik 15.00 kali lipat, yang awalnya 1.04%, naik jadi 15.6%.

Aturan ketiga itu Confidence-nya 66.7%, berarti jika Chilli Lights (79321) dibeli, maka kemungkinan Chilli Lights (79321) dibeli lagi itu 66.7%.

Lift-nya 321.41, berarti jika Chilli Lights (79321) dibeli, maka kemungkinan buat Chilli Lights (79321) dibeli itu naik 321.41 kali lipat, yang awalnya 0.21%, naik jadi 66.7%.